## Nelson-Siegel Yield Curve Fitting

Fits a smooth yield curve to discrete Treasury rate observations from TerraFin's data layer.

**What it does:** Interpolation — not forecasting. Given observed Treasury yields at a few
maturities, Nelson-Siegel produces a continuous curve you can query at any maturity.

**Applications:**
- Maturity-matched discount rates for DCF (instead of a single flat risk-free rate)
- Forward rate computation at arbitrary tenors
- Yield curve shape visualization and inversion detection

### 1. Fetch Treasury Data

In [ ]:
from TerraFin import configure, load_terrafin_config
from TerraFin.data import DataFactory


configure()
config = load_terrafin_config()

data = DataFactory()

# TerraFin provides Treasury yields at 5 maturities
treasury_keys = {
    "Treasury-13W": 0.25,  # 13 weeks = 0.25 years
    "Treasury-2Y": 2.0,
    "Treasury-5Y": 5.0,
    "Treasury-10Y": 10.0,
    "Treasury-30Y": 30.0,
}

# Get latest yield for each maturity
maturities = []
yields = []
for name, mat in treasury_keys.items():
    df = data.get(name)
    latest_yield = df["close"].iloc[-1]
    maturities.append(mat)
    yields.append(latest_yield)
    print(f"{name:>15s}: {latest_yield:.3f}%")

print(f"\nMaturities: {maturities}")
print(f"Yields:     {[round(y, 3) for y in yields]}")

### 2. Fit Nelson-Siegel Curve

In [ ]:
from TerraFin.analytics.analysis.rates import fit


curve = fit(maturities, yields)

print("Fitted parameters:")
print(f"  beta0 (level):     {curve.beta0:.4f}")
print(f"  beta1 (slope):     {curve.beta1:.4f}")
print(f"  beta2 (curvature): {curve.beta2:.4f}")
print(f"  tau (decay):       {curve.tau:.4f}")
print(f"  RMSE:              {curve.rmse:.4f}%")

print("\nFit quality:")
for mat, obs, res in zip(curve.maturities, curve.observed_yields, curve.residuals()):
    print(f"  {mat:5.2f}Y: observed={obs:.3f}%  residual={res:+.4f}%")

### 3. Interpolated Yield Curve

In [ ]:
import numpy as np
import plotly.graph_objects as go


# Smooth curve at fine granularity
fine_maturities = np.linspace(0.1, 30, 300)
fine_yields = curve.yields_at(fine_maturities.tolist())

fig = go.Figure()

# Fitted curve
fig.add_trace(
    go.Scatter(
        x=fine_maturities,
        y=fine_yields,
        mode="lines",
        name="Nelson-Siegel Fit",
        line={"color": "#1f77b4", "width": 2},
    )
)

# Observed points
fig.add_trace(
    go.Scatter(
        x=maturities,
        y=yields,
        mode="markers",
        name="Observed Treasury Yields",
        marker={"size": 10, "color": "#d62728"},
    )
)

fig.update_layout(
    title="Nelson-Siegel Yield Curve",
    xaxis_title="Maturity (years)",
    yaxis_title="Yield (%)",
    template="plotly_white",
    height=450,
)
fig.show()

### 4. Interpolation Examples

Query yields at maturities where we have no direct Treasury data.

In [ ]:
query_maturities = [0.5, 1.0, 3.0, 7.0, 15.0, 20.0]

print("Interpolated yields:")
for m in query_maturities:
    print(f"  {m:5.1f}Y: {curve.yield_at(m):.3f}%")

### 5. Forward Rates

Implied forward rates derived from the fitted curve:
$$f(t_1, t_2) = \frac{y(t_2) \cdot t_2 - y(t_1) \cdot t_1}{t_2 - t_1}$$

In [ ]:
# Common forward rate tenors
forward_pairs = [
    (0.25, 1.0, "3M-1Y"),  # What rate does the market imply for 9M starting in 3M?
    (1.0, 2.0, "1Y-2Y"),
    (2.0, 5.0, "2Y-5Y"),
    (5.0, 10.0, "5Y-10Y"),
    (10.0, 30.0, "10Y-30Y"),
    (1.0, 3.0, "1Y-3Y (18M forward)"),  # Compare with TerraFin's existing 18M forward
]

print("Implied forward rates:")
for t1, t2, label in forward_pairs:
    fwd = curve.forward_rate(t1, t2)
    print(f"  f({label:>20s}): {fwd:.3f}%")

### 6. Application: Maturity-Matched DCF Discount Rates

Instead of a single flat risk-free rate for all cash flow years,
use the fitted curve to get a rate matched to each year.

In [ ]:
# Compare flat rate vs curve-matched discounting
flat_rate = curve.yield_at(10.0)  # Typical: use 10Y as the single risk-free rate

print(f"Flat 10Y rate: {flat_rate:.3f}%\n")
print(f"{'Year':>6s}  {'Curve Rate':>12s}  {'Flat Rate':>12s}  {'Diff':>8s}")
print("-" * 44)
for year in range(1, 6):
    curve_rate = curve.yield_at(float(year))
    diff = curve_rate - flat_rate
    print(f"{year:>6d}  {curve_rate:>11.3f}%  {flat_rate:>11.3f}%  {diff:>+7.3f}%")

### 7. Curve Shape Interpretation

The Nelson-Siegel parameters have direct economic interpretation:
- `beta1 < 0`: inverted curve (short rates > long rates)
- `beta1 > 0`: normal upward-sloping curve
- `|beta2|` large: pronounced hump or dip in the middle
- `beta0`: asymptotic long-run yield level

In [ ]:
# Classify the current curve shape
slope_2s10s = curve.yield_at(10.0) - curve.yield_at(2.0)
slope_3m10y = curve.yield_at(10.0) - curve.yield_at(0.25)

print("Curve shape diagnostics:")
print(f"  2s10s spread:  {slope_2s10s:+.3f}%")
print(f"  3m10y spread:  {slope_3m10y:+.3f}%")
print(f"  beta1 (slope): {curve.beta1:+.4f} {'(inverted)' if curve.beta1 < 0 else '(normal)'}")
print(
    f"  beta2 (curve): {curve.beta2:+.4f} {'(humped)' if curve.beta2 > 0.5 else '(flat mid-section)' if abs(curve.beta2) < 0.5 else '(dipped)'}"
)